In [1]:
import torch
torch.cuda.is_available()

True

In [2]:
tt = "Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in \"Harry Potter and the Order of the Phoenix\" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. \"I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar,\" he told an Australian interviewer earlier this month. \"I don't think I'll be particularly extravagant. \"The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs.\" At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film \"Hostel: Part II,\" currently six places below his number one movie on the UK box office chart. Details of how he'll mark his landmark birthday are under wraps. His agent and publicist had no comment on his plans. \"I'll definitely have some sort of party,\" he said in an interview. \"Hopefully none of you will be reading about it.\" Radcliffe's earnings from the first five Potter films have been held in a trust fund which he has not been able to touch. Despite his growing fame and riches, the actor says he is keeping his feet firmly on the ground. \"People are always looking to say 'kid star goes off the rails,'\" he told reporters last month. \"But I try very hard not to go that way because it would be too easy for them.\" His latest outing as the boy wizard in \"Harry Potter and the Order of the Phoenix\" is breaking records on both sides of the Atlantic and he will reprise the role in the last two films. Watch I-Reporter give her review of Potter's latest » . There is life beyond Potter, however. The Londoner has filmed a TV movie called \"My Boy Jack,\" about author Rudyard Kipling and his son, due for release later this year. He will also appear in \"December Boys,\" an Australian film about four boys who escape an orphanage. Earlier this year, he made his stage debut playing a tortured teenager in Peter Shaffer's \"Equus.\" Meanwhile, he is braced for even closer media scrutiny now that he's legally an adult: \"I just think I'm going to be more sort of fair game,\" he told Reuters."

In [3]:
from transformers import BartTokenizer, BartForConditionalGeneration, BartConfig
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)
model = BartForConditionalGeneration.from_pretrained('sshleifer/distilbart-cnn-12-6').to(device)
tokenizer = BartTokenizer.from_pretrained('sshleifer/distilbart-cnn-12-6')

cuda:0


In [27]:
# https://huggingface.co/blog/how-to-generate
# https://huggingface.co/blog/constrained-beam-search
# https://github.com/huggingface/transformers/issues/2704
# Greedy search and Beam search are known to not generate text well.
# Top-K sampling and Top-p sampling seems better performed.
#
# In Hugging face model generate, there are multiple modes:
#   - `is_greedy_gen_mode` stands for Greedy Search.
#     Require:
#        + num_beams == 1
#        + num_beam_groups == 1
#        + do_sample == False
#
#   - `is_beam_gen_mode` stands for Beam Search.
#     Require:
#        + num_beams > 1
#        + num_beam_groups == 1
#        + do_sample == False
#
#   - is_sample_gen_mode stands for Multinomial Sampling (with top_k and top_p).
def summarize(text, 
    reduction_ratio=0.5, 
    top_p=1.0, 
    temperature=1.0, 
    repetition_penalty=1.0, 
    length_penalty=1.0):
    inputs = tokenizer([text], return_tensors='pt', truncation=True).to(device)
    
    # determine min and max length based on the input text length
    # and reduction_ratio.
    input_ids_seq_length = inputs['input_ids'].shape[-1]

    # compute target token based on reduction ratio:
    max_new_tokens = input_ids_seq_length * (1 - reduction_ratio)
    if max_new_tokens + input_ids_seq_length <= 10:
        max_new_tokens = 10

    summary_ids = model.generate(inputs['input_ids'],
        # Define stopping criteria.
        max_time=5, # allow maximum 5 seconds of inference.

        # Set these parameters to use TopP sampling.
        do_sample=True,
        num_beams=1,
        num_beam_groups=1,
        # The number of highest probability vocabulary tokens to keep for
        # top-k-filtering. Default 1.0, and no upper bound.
        # We want to disable this because limiting the sample pool to a
        # fixed size K could endanger the model to produce gibberish for
        # sharp distributions and limit the model's creativity for flat
        # distribution. See https://arxiv.org/abs/1904.09751.
        top_k=0,

        # determine length
        min_length=10,
        max_length=None,
        max_new_tokens=max_new_tokens,

        # If set to float < 1, only the most probable tokens with
        # probabilities that add up to `top_p` or higher are kept for
        # generation. Default 1.0, from 0 to 1.
        top_p=top_p,
        # The value used to module the next token probabilities.
        # Default 1.0, from 0 to 1.
        temperature=temperature,
        # The parameter for repetition penalty. 1.0 means no penalty.
        # See https://arxiv.org/pdf/1909.05858.pdf for more details.
        # Default 1.0.
        repetition_penalty=repetition_penalty,
        # Exponential penalty to the length. 1.0 means that the beam
        # score is penalized by the sequence length. 0.0 means no penalty.
        # Set to values < 0.0 in order to encourage the model to generate
        # longer sequences, to a value > 0.0 in order to encourage the
        # model to produce shorter sequences. Default 1.0.
        length_penalty=length_penalty,

        # We desire one outcome per parameter set.
        num_return_sequences=1,
    )
    return [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in summary_ids]

import random
import numpy as np

# For reproducible results.
seed = 42
random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

for i in range(1, 10):
    print(f'reduction_ratio = {i/10}')
    for t in summarize(tt, reduction_ratio=0.5, top_p=1, temperature=0.5, repetition_penalty=1, length_penalty=i/10):
        print(t)

reduction_ratio = 0.1
 Daniel Radcliffe turns 18 on Monday, gaining access to a reported $41.1 million fortune. Harry Potter star says he won't fritter his cash away on fast cars, drink and celebrity parties. Harry Harry Potter actor says he will not spend much of his fortune on frivolous purchases.
reduction_ratio = 0.2
 Harry Potter actor Daniel Radcliffe turns 18 on Monday. Radcliffe has a reported $41.1 million fortune as he turns 18. The young actor says he has no plans to fritter his fortune away on fast cars and drink. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he says.
reduction_ratio = 0.3
 Harry Potter star Daniel Radcliffe turns 18 on Monday, gaining access to a reported $41.1 million fortune. The young actor says he has no plans to fritter his money away on fast cars, drink and celebrity parties. Radcliffe's earnings from the first five Potter films have been held in a 

Harry Potter star Daniel Radcliffe turns 18 on Monday, gaining access to a reported $41.1 million fortune . Radcliffe says he has no plans to fritter his cash away on fast cars, drink and celebrity parties . The Londoner's earnings from the first five Potter films have been held in a trust fund .

In [ ]:
from transformers import BartTokenizer, BartModel
import torch
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
model = BartModel.from_pretrained('facebook/bart-large-cnn')